# Results Analysis — Credit Card Fraud (Autoencoder Anomaly Detection)

**Dissertation:** Graph Neural Networks for Systemic Risk and Fraud Detection in Credit Systems
**Track:** Fraud — unsupervised **autoencoder** anomaly detection (Objective 3)
**Student:** Chandan Nagavolu — M.Tech, BITS Pilani WILP

The ULB Credit Card dataset (~284,807 transactions, **0.172% fraud**) is the
classic reconstruction-error benchmark. An **autoencoder is trained on normal
transactions only**; a transaction's reconstruction error is its anomaly score,
because fraud — unlike anything the AE learned — reconstructs poorly. This is an
*unsupervised early-warning* detector, benchmarked against *supervised* baselines
(Objective 4).

Reads `reports/results_creditcard.csv` and the AE analysis under
`reports/creditcard/` + `reports/figures/`, produced by
`run_creditcard_pipeline.py`.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110


def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "CLAUDE.md").exists():
            return cand
    raise FileNotFoundError("Could not locate project root (CLAUDE.md).")


ROOT = find_project_root(Path.cwd())
RESULTS_CSV = ROOT / "reports" / "results_creditcard.csv"
CC_DIR = ROOT / "reports" / "creditcard"
FIG_DIR = ROOT / "reports" / "figures"

MODEL_ORDER = ["logistic_regression", "random_forest", "xgboost", "autoencoder"]
MODEL_PRETTY = {"logistic_regression": "Logistic Regression", "random_forest": "Random Forest",
                "xgboost": "XGBoost", "autoencoder": "Autoencoder (unsupervised)"}
METRICS = ["pr_auc", "mcc", "f1", "roc_auc", "ks_statistic", "gini"]
PRETTY = {"pr_auc": "PR-AUC", "mcc": "MCC", "f1": "F1", "roc_auc": "ROC-AUC",
          "ks_statistic": "KS", "gini": "Gini"}
HAVE_RESULTS = RESULTS_CSV.exists()
print("Project root:", ROOT, "| results present:", HAVE_RESULTS)


## 1. Why these metrics?

At 0.172% fraud, **ROC-AUC is over-optimistic** — a model can look near-perfect
while missing most fraud. The headline metrics are therefore:

| Metric | Why it matters |
|---|---|
| **PR-AUC** | Area under precision-recall; the right summary for a rare positive class. |
| **MCC** | Balanced score over all four confusion cells; robust to imbalance. |
| **F1** | Harmonic mean of precision/recall at the chosen threshold. |
| **KS / Gini** | Rank-ordering / separation, reported for cross-track consistency. |

The autoencoder is scored on its **reconstruction error** (ranking metrics) with
a threshold tuned on validation (F1/MCC).

In [ ]:
if HAVE_RESULTS:
    raw = pd.read_csv(RESULTS_CSV, parse_dates=["timestamp"]).sort_values("timestamp")
    latest = raw.groupby(["model", "split"], as_index=False).tail(1).reset_index(drop=True)
    display(latest[["model", "split", *METRICS]].round(4))
else:
    print("Run run_creditcard_pipeline.py first.")

## 2. Leaderboard — autoencoder vs supervised baselines (test)

The Objective-4 comparison. The supervised models were trained on labelled fraud;
the autoencoder never saw a fraud label. The question is how close an
*unsupervised* anomaly detector gets to *supervised* performance — important
because in production, labelled fraud is scarce and arrives late.

In [ ]:
if HAVE_RESULTS:
    board = latest[latest.split == "test"].set_index("model").reindex(MODEL_ORDER).dropna(how="all")
    tbl = board[METRICS].rename(columns=PRETTY); tbl.index = [MODEL_PRETTY.get(m, m) for m in tbl.index]
    display(tbl.round(4))
    present = [m for m in MODEL_ORDER if m in board.index]
    fig, ax = plt.subplots(figsize=(12, 6)); x = np.arange(len(METRICS)); w = 0.8/max(len(present),1)
    for i, m in enumerate(present):
        ax.bar(x + (i-(len(present)-1)/2)*w, board.loc[m, METRICS].values.astype(float), w,
               label=MODEL_PRETTY.get(m, m), edgecolor="white")
    ax.set_xticks(x); ax.set_xticklabels([PRETTY[m] for m in METRICS]); ax.set_ylim(0, 1.05)
    ax.set_title("Credit Card — autoencoder vs supervised baselines (test)"); ax.set_ylabel("Score")
    ax.legend(fontsize=9); ax.grid(True, axis="y", alpha=0.3); fig.tight_layout(); plt.show()

## 3. Reconstruction-error distribution

The intuition behind the whole approach: normal transactions reconstruct with
low error, fraud with high error. The tuned threshold (dashed) is where the
detector draws the line. Clear separation here is what makes the unsupervised
detector viable.

In [ ]:
img = FIG_DIR / "creditcard_recon_error_dist.png"
if img.exists():
    fig, ax = plt.subplots(figsize=(10, 5)); ax.imshow(mpimg.imread(img)); ax.axis("off")
    fig.tight_layout(); plt.show()
else:
    print("Run src/evaluation/evaluate_creditcard.py first.")

## 4. Precision-recall curve & threshold sweep

In [ ]:
img = FIG_DIR / "creditcard_pr_curve.png"
if img.exists():
    fig, ax = plt.subplots(figsize=(7, 5)); ax.imshow(mpimg.imread(img)); ax.axis("off")
    fig.tight_layout(); plt.show()
sweep_path = CC_DIR / "threshold_sweep.csv"
if sweep_path.exists():
    sweep = pd.read_csv(sweep_path).dropna()
    best = sweep.loc[sweep["f1"].idxmax()]
    print(f"Best-F1 operating point: threshold={best['threshold']:.4f}  "
          f"precision={best['precision']:.3f}  recall={best['recall']:.3f}  f1={best['f1']:.3f}")
    display(sweep.iloc[::max(len(sweep)//12, 1)].round(4))

## 5. Takeaways

- **Unsupervised, yet competitive.** Trained on normal traffic only, the
  autoencoder ranks fraud well (high PR-AUC) and, at a tuned threshold, lands a
  respectable F1/MCC — without ever seeing a fraud label.
- **Supervised still wins when labels exist.** The baselines edge the AE, as
  expected; the AE's value is as an *early-warning* layer for novel fraud before
  labels are available (and as a complement, not a replacement).
- **Threshold is a business choice.** The error distribution + sweep make the
  precision/recall trade-off explicit so a risk team can set the alert rate.
- **Same harness as every track.** The AE is scored through `compute_all_metrics`
  and logged to the shared table, so it compares directly with the GNN and TCN
  models elsewhere in the dissertation.